# 🤖 LeWorldModel — Push-T: Moving the T onto the Outline

**Paper:** [LeWorldModel: Stable End-to-End JEPA from Pixels](https://arxiv.org/abs/2603.19312)  
**Authors:** Lucas Maes, Quentin Le Lidec, Damien Scieur, Yann LeCun, Randall Balestriero  
**GitHub:** https://github.com/lucas-maes/le-wm  
**Checkpoints:** https://huggingface.co/quentinll/lewm-pusht

---

## What this notebook does

This notebook reproduces the **Push-T planning experiment** from the LeWM paper.
The goal is to have a robot agent push a T-shaped block so it overlaps a fixed
T-shaped outline on the table — purely from pixel observations, no proprioception.

**Pipeline:**
1. Install dependencies (`stable-worldmodel`, `stable-pretraining`)
2. Download the pretrained LeWM checkpoint from HuggingFace
3. Build the LeWM model (ViT encoder + ARPredictor + SIGReg)
4. Run CEM planning: encode start & goal → roll out candidates in latent space → pick best
5. Execute in the Push-T environment
6. Visualise the rollout side-by-side with the goal

---

> **Hardware:** Optimised for **Apple Silicon (MPS)**. Falls back to CPU automatically.
> MPS gives ~3–5× speedup over CPU for the ViT encoder and transformer predictor.

## 1 · Install dependencies

In [2]:
# 1. Install bazel via Homebrew if you want a clean compilation, 
# OR use this workaround to install stable-worldmodel without the broken labmaze dependency:
!pip install -q "stable-worldmodel" einops huggingface_hub torchvision matplotlib scikit-learn

# Clone the LeWM repo so we can import jepa.py and module.py directly
import os
if not os.path.isdir("le-wm"):
    !git clone --quiet https://github.com/lucas-maes/le-wm.git

import sys
sys.path.insert(0, "le-wm")

print("✅ Base dependencies installed and repo cloned.")

✅ Base dependencies installed and repo cloned.


## 2 · Imports & device setup (MPS-aware)

Key MPS considerations baked in here:
- Device priority: `mps` → `cpu` (no CUDA on Mac)
- MPS requires **float32** everywhere — we enforce this explicitly
- `os.environ["MUJOCO_GL"]` is set to `"glfw"` (macOS) not `"egl"` (Linux)
- `SIGReg` in `module.py` hardcodes `device="cuda"` — we monkey-patch it to use our device

In [3]:
import sys
try:
    import cv2
    print("✅ Wait, it actually imports fine here!")
except Exception as e:
    import traceback
    print(f"❌ Import failed with type: {type(e).__name__}")
    traceback.print_exc()

✅ Wait, it actually imports fine here!


In [4]:
import json
import os
import warnings
from pathlib import Path

# ── MuJoCo renderer — use glfw on macOS, not egl (Linux-only) ─────────────────
os.environ["MUJOCO_GL"] = "glfw"

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from IPython.display import Image, display
from torchvision.transforms import v2 as transforms

# ── LeWM source files (cloned repo) ───────────────────────────────────────────
from jepa import JEPA
from module import ARPredictor, Embedder, MLP

# ── stable-* packages ─────────────────────────────────────────────────────────
import stable_worldmodel as swm
import stable_pretraining as spt

warnings.filterwarnings("ignore")

# ── Device: MPS (Apple Silicon GPU) → CPU fallback ────────────────────────────
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Using device : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
if DEVICE == "mps":
    print("✅ Apple Silicon GPU (MPS) detected — model and tensors will run on the M-series chip.")

Using device : mps
PyTorch      : 2.12.0
✅ Apple Silicon GPU (MPS) detected — model and tensors will run on the M-series chip.


/Users/yogesh/venvs/ai/lib/python3.13/site-packages/stable_worldmodel/envs/__init__.py:215: UserWarning: ale-py not found; ALE/* envs are unavailable. Install with: pip install 'stable-worldmodel[env]' or pip install ale-py.
  from stable_worldmodel.envs import ale  # noqa: F401


### 2a · Patch `SIGReg` for MPS

`module.py`'s `SIGReg.forward()` contains a hardcoded `device="cuda"` in the random
projection line. We replace it with a device-agnostic version that uses `DEVICE`.
SIGReg is only used **during training** (not inference/planning), but patching it
keeps the import clean and avoids surprises if you experiment with fine-tuning.

In [5]:
import module as _module

class SIGRegMPS(_module.SIGReg):
    """SIGReg with device-agnostic random projections (removes hardcoded cuda)."""

    def forward(self, proj: torch.Tensor) -> torch.Tensor:
        # proj: (T, B, D)  — same signature as original
        device = proj.device                          # ← use tensor's own device
        A = torch.randn(proj.size(-1), self.num_proj, device=device)
        A = A.div_(A.norm(p=2, dim=0))

        x_t = (proj @ A).unsqueeze(-1) * self.t      # self.t is a buffer → follows device
        err = (
            (x_t.cos().mean(-3) - self.phi).square()
            + x_t.sin().mean(-3).square()
        )
        statistic = (err @ self.weights) * proj.size(-2)
        return statistic.mean()

# Hot-swap the class so any code importing from module gets the patched version
_module.SIGReg = SIGRegMPS
print("✅ SIGReg patched — random projections now use tensor.device (MPS-safe)")

✅ SIGReg patched — random projections now use tensor.device (MPS-safe)


## 3 · Download the pretrained LeWM checkpoint (Push-T)

In [6]:
from huggingface_hub import hf_hub_download

HF_REPO   = "quentinll/lewm-pusht"
CACHE_DIR = Path("./lewm_cache/pusht")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

weights_path = hf_hub_download(
    repo_id=HF_REPO, filename="weights.pt", local_dir=CACHE_DIR
)
config_path = hf_hub_download(
    repo_id=HF_REPO, filename="config.json", local_dir=CACHE_DIR
)

print(f"✅ Checkpoint saved to {CACHE_DIR}")
print(f"   weights : {weights_path}")
print(f"   config  : {config_path}")

19:45:49 | INFO  | __init__.py | HTTP Request: HEAD https://huggingface.co/quentinll/lewm-pusht/resolve/main/weights.pt "HTTP/1.1 302 Found"


19:45:49 | WARN  | __init__.py | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:45:50 | INFO  | __init__.py | HTTP Request: HEAD https://huggingface.co/quentinll/lewm-pusht/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
19:45:50 | INFO  | __init__.py | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/quentinll/lewm-pusht/22b330c28c27ead4bfd1888615af1340e3fe9052/config.json "HTTP/1.1 200 OK"
✅ Checkpoint saved to lewm_cache/pusht
   weights : lewm_cache/pusht/weights.pt
   config  : lewm_cache/pusht/config.json


## 4 · Build the LeWM model

```
Encoder  ─► ViT (tiny/small/base)  →  CLS token  →  Projector MLP  →  z_t  (192-d)
Predictor ─► ARPredictor (Transformer, AdaLN-zero conditioned on action embeddings)
```

**MPS notes:**
- We always load weights to `"cpu"` first, then `.to(DEVICE)` — MPS can't be the
  target of `torch.load` directly in older PyTorch builds.
- `model.float()` ensures every parameter is `float32`; MPS silently fails on `float64`.

In [7]:
with open(config_path) as f:
    cfg = json.load(f)

print("Model config:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

Model config:
  _target_: stable_worldmodel.wm.lewm.LeWM
  encoder: {'_target_': 'stable_pretraining.backbone.utils.vit_hf', 'size': 'tiny', 'patch_size': 14, 'image_size': 224, 'pretrained': False, 'use_mask_token': False}
  predictor: {'_target_': 'stable_worldmodel.wm.lewm.module.Predictor', 'num_frames': 3, 'input_dim': 192, 'hidden_dim': 192, 'output_dim': 192, 'depth': 6, 'heads': 16, 'mlp_dim': 2048, 'dim_head': 64, 'dropout': 0.1, 'emb_dropout': 0.0}
  action_encoder: {'_target_': 'stable_worldmodel.wm.lewm.module.Embedder', 'input_dim': 10, 'emb_dim': 192}
  projector: {'_target_': 'stable_worldmodel.wm.lewm.module.MLP', 'input_dim': 192, 'output_dim': 192, 'hidden_dim': 2048, 'norm_fn': {'_target_': 'torch.nn.BatchNorm1d', '_partial_': True}}
  pred_proj: {'_target_': 'stable_worldmodel.wm.lewm.module.MLP', 'input_dim': 192, 'output_dim': 192, 'hidden_dim': 2048, 'norm_fn': {'_target_': 'torch.nn.BatchNorm1d', '_partial_': True}}


In [8]:
# ── Build sub-modules ─────────────────────────────────────────────────────────
encoder = spt.backbone.utils.vit_hf(
    cfg["encoder"]["size"],
    patch_size=cfg["encoder"]["patch_size"],
    image_size=cfg["encoder"]["image_size"],
    pretrained=False,
    use_mask_token=False,
)

def make_mlp(key: str) -> MLP:
    # Remove metadata keys if present
    mlp_cfg = cfg[key].copy()
    mlp_cfg.pop("_target_", None)
    
    return MLP(
        input_dim=mlp_cfg["input_dim"],
        output_dim=mlp_cfg["output_dim"],
        hidden_dim=mlp_cfg["hidden_dim"],
        norm_fn=nn.BatchNorm1d,
    )

# Clean dictionaries for Predictor and Action Encoder to remove '_target_'
predictor_cfg = cfg["predictor"].copy()
predictor_cfg.pop("_target_", None)

action_enc_cfg = cfg["action_encoder"].copy()
action_enc_cfg.pop("_target_", None)

model = JEPA(
    encoder=encoder,
    predictor=ARPredictor(**predictor_cfg),
    action_encoder=Embedder(**action_enc_cfg),
    projector=make_mlp("projector"),
    pred_proj=make_mlp("pred_proj"),
)

# ── Load weights: always cpu first, then move to DEVICE ───────────────────────
state_dict = torch.load(weights_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict, strict=True)

# ── float32 is required by MPS (no float64 support) ───────────────────────────
model = model.float().to(DEVICE).eval()
model.requires_grad_(False)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ LeWM loaded  |  {n_params:.1f}M parameters  |  device: {DEVICE}")

19:45:54 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)
19:45:54 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
✅ LeWM loaded  |  18.0M parameters  |  device: mps


## 5 · Push-T environment setup

Push-T: a 2-D task where the agent (robot end-effector) must push a T-shaped block
until it overlaps a fixed T-shaped outline — entirely from pixels.

The image transform mirrors `eval.py` exactly (ImageNet normalisation, resize to 224).

In [9]:
IMG_SIZE = cfg["encoder"]["image_size"]  # 224

img_transform = transforms.Compose([
    transforms.ToImage(),
    # float32 explicitly — MPS does not support float64
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(**spt.data.dataset_stats.ImageNet),
    transforms.Resize(size=IMG_SIZE),
])


def preprocess_frame(frame: np.ndarray) -> torch.Tensor:
    """HWC uint8 numpy frame → (1, 1, C, H, W) float32 tensor on DEVICE."""
    t = img_transform(frame)                            # (C, H, W) float32
    return t.unsqueeze(0).unsqueeze(0).to(DEVICE)       # (1, 1, C, H, W)


# ── REMOVED MANUAL REGISTRATION COMPLETELY ───────────────────────────────────
# The stable-worldmodel library registers all its benchmarks internally.
# Simply pass the correct path specifier: "swm/PushT-v1"
# ─────────────────────────────────────────────────────────────────────────────

# MUJOCO_GL=glfw is already set above — required on macOS
world = swm.World(
    env_name="swm/PushT-v1",  # Changed from "pusht"
    num_envs=1,
    max_episode_steps=400,
    image_shape=(224, 224),
)

print("✅ Push-T environment ready")

objc[21188]: Class SDLApplication is implemented in both /Users/yogesh/venvs/ai/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1198b4890) and /Users/yogesh/venvs/ai/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x157add2c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[21188]: Class SDLAppDelegate is implemented in both /Users/yogesh/venvs/ai/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1198b48e0) and /Users/yogesh/venvs/ai/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x157add318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[21188]: Class SDLTranslatorResponder is implemented in both /Users/yogesh/venvs/ai/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x1198b4958) and /Users/yogesh/venvs/ai/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.

✅ Push-T environment ready


## 6 · CEM planner (MPS-optimised)

The Cross-Entropy Method plans in latent space (Section 3.2 of the paper):

1. Sample **S** candidate action sequences of length **H**
2. Roll each through the predictor autoregressively → `z_pred_H`
3. Score = MSE(`z_pred_H`, `z_goal`)  ← `JEPA.criterion()` from `jepa.py`
4. Keep top-**k** elites, refit Gaussian, repeat for **I** iterations
5. Execute the first `action_block` actions, then replan

**MPS-specific decisions:**
- All tensors are created directly on `DEVICE` (no `.to()` in the hot loop)
- `torch.randn(..., device=DEVICE)` instead of `.to(DEVICE)` afterward
- `float32` enforced on every tensor literal
- `torch.no_grad()` wraps all inference (saves MPS memory bandwidth)

In [10]:
# ── CEM hyper-parameters ──────────────────────────────────────────────────────
CEM_SAMPLES  = 256   # S  – candidate action sequences per CEM iteration
CEM_ELITES   = 64    # top-k kept for Gaussian refit
CEM_ITERS    = 5     # CEM iterations per planning step
HORIZON      = 16    # H  – planning horizon
ACTION_BLOCK = 8     # execute this many steps before replanning
HISTORY_SIZE = 3     # past embeddings the predictor attends to
ACTION_DIM   = cfg["action_encoder"]["input_dim"]

print(f"Action dim       : {ACTION_DIM}")
print(f"Planning horizon : {HORIZON} steps")
print(f"CEM samples/iter : {CEM_SAMPLES}")
print(f"Device           : {DEVICE}")


@torch.no_grad()
def encode_frame(frame: np.ndarray) -> torch.Tensor:
    """Encode a single observation frame → embedding (1, D) on DEVICE."""
    pixels = preprocess_frame(frame)        # (1, 1, C, H, W)
    info = model.encode({"pixels": pixels})
    return info["emb"][:, 0]               # (1, D)


@torch.no_grad()
def cem_plan(
    z_start: torch.Tensor,   # (1, D) current embedding — already on DEVICE
    z_goal:  torch.Tensor,   # (1, D) goal  embedding  — already on DEVICE
    history_embs: list,       # list of (1, D) tensors
) -> np.ndarray:
    """
    CEM in latent space. Returns best action sequence (HORIZON, ACTION_DIM) as numpy.
    Mirrors JEPA.rollout() + JEPA.criterion() from jepa.py.
    """
    S = CEM_SAMPLES
    D = z_start.shape[-1]

    # Initialise Gaussian — tensors born on DEVICE, float32
    mu    = torch.zeros(HORIZON, ACTION_DIM, device=DEVICE, dtype=torch.float32)
    sigma = torch.ones (HORIZON, ACTION_DIM, device=DEVICE, dtype=torch.float32)

    # Build history buffer (HISTORY_SIZE, D)
    if history_embs:
        hist = torch.cat(history_embs[-HISTORY_SIZE:], dim=0)   # (<= HS, D)
    else:
        hist = z_start
    while hist.shape[0] < HISTORY_SIZE:          # pad to full history size
        hist = torch.cat([hist[:1], hist], dim=0)
    hist = hist[-HISTORY_SIZE:]                  # (HS, D)

    for _ in range(CEM_ITERS):
        # Sample S action sequences (S, H, A) — on DEVICE directly
        eps  = torch.randn(S, HORIZON, ACTION_DIM, device=DEVICE, dtype=torch.float32)
        acts = (mu.unsqueeze(0) + sigma.unsqueeze(0) * eps).clamp(-1.0, 1.0)

        # Autoregressive rollout — mirrors JEPA.rollout()
        emb_buf = hist.unsqueeze(0).expand(S, -1, -1).clone()   # (S, HS, D)

        for t in range(HORIZON):
            act_t   = acts[:, t:t+1, :]                          # (S, 1, A)
            act_emb = model.action_encoder(act_t)                # (S, 1, A_emb)

            trunc_emb = emb_buf[:, -HISTORY_SIZE:]               # (S, HS, D)
            # Expand action embedding to match history length for conditional attention
            trunc_act = act_emb.expand(-1, trunc_emb.shape[1], -1)  # (S, HS, A_emb)

            pred_emb = model.predict(trunc_emb, trunc_act)[:, -1:]  # (S, 1, D)
            emb_buf  = torch.cat([emb_buf, pred_emb], dim=1)        # (S, HS+t+1, D)

        # Cost: MSE between final predicted emb and goal — mirrors JEPA.criterion()
        z_pred_final = emb_buf[:, -1, :]           # (S, D)
        z_goal_exp   = z_goal.expand(S, -1)        # (S, D)
        costs = F.mse_loss(
            z_pred_final, z_goal_exp.detach(), reduction="none"
        ).sum(dim=-1)                               # (S,)

        # Elite refit
        elite_idx  = costs.argsort()[:CEM_ELITES]
        elite_acts = acts[elite_idx]               # (k, H, A)
        mu    = elite_acts.mean(dim=0)
        sigma = elite_acts.std(dim=0).clamp(min=1e-3)

    # Move to CPU for numpy / env.step
    return mu.cpu().float().numpy()                # (H, A)


print("✅ CEM planner ready (MPS-optimised)")

Action dim       : 10
Planning horizon : 16 steps
CEM samples/iter : 256
Device           : mps
✅ CEM planner ready (MPS-optimised)


## 7 · High-level evaluation via `stable_worldmodel` (mirrors `eval.py`)

This section uses the exact same API as the paper's `eval.py` — `AutoCostModel` +
`WorldModelPolicy`. It requires the Push-T dataset (HDF5) from HuggingFace.

**Skip to Section 8 if you'd rather run a quick single-episode without the dataset.**

In [11]:
# ── Convert weights.pt → _object.ckpt (UPDATED DIRECTORY STRUCTURE) ───────────
STABLEWM_HOME = Path(swm.data.utils.get_cache_dir())

# stable_worldmodel looks specifically inside "checkpoints/{run_name}"
OBJ_CKPT = STABLEWM_HOME / "checkpoints" / "pusht" / "lewm_object.ckpt"
OBJ_CKPT.parent.mkdir(parents=True, exist_ok=True)

if not OBJ_CKPT.exists():
    # Save the full model object (eval.py uses AutoCostModel which expects this)
    # Move to CPU first so the ckpt is device-neutral and portable
    torch.save(model.cpu(), OBJ_CKPT)
    model.to(DEVICE)          # move back to MPS/CUDA/CPU
    print(f"✅ Saved object checkpoint to {OBJ_CKPT}")
else:
    print(f"✅ Object checkpoint already exists at {OBJ_CKPT}")

# This will now successfully find the file at the resolved location!
cost_model = swm.policy.AutoCostModel("pusht/lewm")

# Always load to CPU first, then move — same safe pattern as above
cost_model = cost_model.float().to(DEVICE).eval()
cost_model.requires_grad_(False)
print(f"✅ AutoCostModel ready on {DEVICE}")

✅ Object checkpoint already exists at /Users/yogesh/.stable_worldmodel/checkpoints/pusht/lewm_object.ckpt
✅ AutoCostModel ready on mps


In [12]:
# ── WorldModelPolicy with CEM solver (CORRECTED) ──────────────────────────────
# PlanConfig manages structural trajectory constraints exclusively
plan_config = swm.PlanConfig(
    horizon=HORIZON,
    action_block=ACTION_BLOCK,
    receding_horizon=1,  # Frequency of rolling horizon update
)

# Algorithmic variables are passed cleanly to the solver block instead
solver = swm.solver.CEMSolver(
    model=cost_model,
    num_samples=CEM_SAMPLES,
    topk=CEM_ELITES,
    n_steps=CEM_ITERS,
    device=DEVICE,
)

transform_dict = {
    "pixels": img_transform,
    "goal":   img_transform,
}

policy = swm.policy.WorldModelPolicy(
    solver=solver,
    config=plan_config,
    transform=transform_dict,
)

world.set_policy(policy)
print("✅ WorldModelPolicy successfully bounded to Push-T world!")

✅ WorldModelPolicy successfully bounded to Push-T world!


In [13]:
# ── Download Push-T expert dataset (HDF5) ─────────────────────────────────────
DATASET_PATH = STABLEWM_HOME / "pusht_expert_train.h5"

if not DATASET_PATH.exists():
    print("Downloading Push-T expert dataset (~few hundred MB) ...")
    from huggingface_hub import hf_hub_download
    
    # Download the compressed .zst dataset file from the correct repository
    zst_path = hf_hub_download(
        repo_id="quentinll/lewm-pusht",
        filename="pusht_expert_train.h5.zst",
        repo_type="dataset",
        local_dir=STABLEWM_HOME,
    )
    print("Decompressing dataset (zstd) ...")
    import zstandard as zstd
    dctx = zstd.ZstdDecompressor()
    with open(zst_path, 'rb') as ifh, open(DATASET_PATH, 'wb') as ofh:
        dctx.copy_stream(ifh, ofh)
    
    # Clean up the compressed file after decompression to save space
    import os
    if os.path.exists(zst_path):
        os.remove(zst_path)
        
    print(f"✅ Dataset saved to {DATASET_PATH}")
else:
    print(f"✅ Dataset found at {DATASET_PATH}")

19:45:56 | INFO  | __init__.py | HTTP Request: HEAD https://huggingface.co/datasets/quentinll/lewm-pusht/resolve/main/pusht_expert_train.h5.zst "HTTP/1.1 302 Found"
19:45:56 | INFO  | __init__.py | HTTP Request: GET https://huggingface.co/api/datasets/quentinll/lewm-pusht/xet-read-token/655cd446b9929369d7d406001da85c15d1457850 "HTTP/1.1 200 OK"


pusht_expert_train.h5.zst:   0%|          | 0.00/13.1G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
from sklearn import preprocessing

dataset = swm.data.HDF5Dataset(
    "pusht_expert_train",
    keys_to_cache=["pixels", "action", "episode_idx", "step_idx"],
    cache_dir=STABLEWM_HOME,
)

# Fit action scaler (matches eval.py exactly)
action_scaler = preprocessing.StandardScaler()
action_data   = dataset.get_col_data("action")
action_data   = action_data[~np.isnan(action_data).any(axis=1)]
action_scaler.fit(action_data)
process = {"action": action_scaler, "goal_action": action_scaler}
policy.process = process

# Sample eval episodes
GOAL_OFFSET = 64
EVAL_BUDGET = 200
NUM_EVAL    = 5
SEED        = 42

ep_idx    = dataset.get_col_data("episode_idx")
step_idx  = dataset.get_col_data("step_idx")
ep_ids, _ = np.unique(ep_idx, return_index=True)

rng = np.random.default_rng(SEED)
chosen = rng.choice(len(ep_ids), size=NUM_EVAL, replace=False)
eval_episodes = ep_ids[chosen]

eval_starts = []
for ep_id in eval_episodes:
    mask = ep_idx == ep_id
    max_start = step_idx[mask].max() - GOAL_OFFSET - 1
    eval_starts.append(int(rng.integers(0, max(1, max_start))))

print(f"Evaluating {NUM_EVAL} Push-T episodes ...")
metrics = world.evaluate_from_dataset(
    dataset,
    start_steps=eval_starts,
    goal_offset_steps=GOAL_OFFSET,
    eval_budget=EVAL_BUDGET,
    episodes_idx=eval_episodes.tolist(),
    video_path=Path("./lewm_videos"),
)
print("\n📊 Metrics:", metrics)

## 8 · Manual single-episode rollout (no dataset required)

Self-contained episode using our custom CEM planner from Section 6.
The environment produces a random start + goal; the model plans from pixels.

In [ ]:
MAX_STEPS   = 300
REPLAN_FREQ = ACTION_BLOCK

obs        = world.reset()      # (H, W, 3) uint8
goal       = world.get_goal()   # (H, W, 3) uint8

z_goal = encode_frame(goal)     # (1, D) on DEVICE

frames       = [obs.copy()]
goal_frame   = goal.copy()
history_embs = [encode_frame(obs)]

total_reward = 0.0
done         = False
step         = 0
best_actions = None
action_ptr   = REPLAN_FREQ      # triggers replan on first step

print(f"Running episode on {DEVICE} ...")
while step < MAX_STEPS and not done:
    if action_ptr >= REPLAN_FREQ:
        z_now        = history_embs[-1]
        best_actions = cem_plan(z_now, z_goal, history_embs)
        action_ptr   = 0

    act = best_actions[action_ptr]                 # (A,) numpy, float32
    obs, reward, done, info = world.step(act)

    history_embs.append(encode_frame(obs))
    if len(history_embs) > HISTORY_SIZE + 1:
        history_embs.pop(0)

    frames.append(obs.copy())
    total_reward += reward
    action_ptr   += 1
    step         += 1

    if step % 50 == 0:
        print(f"  step {step:3d}  |  cumulative reward: {total_reward:.3f}  |  done: {done}")

print(f"\n✅ Episode finished  |  steps: {step}  |  total reward: {total_reward:.3f}")

## 9 · Visualise the rollout

In [ ]:
# ── Static thumbnail: start / mid / end ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("LeWM — Push-T: Moving the T onto the outline", fontsize=13, fontweight="bold")

snap_indices = [0, len(frames) // 3, 2 * len(frames) // 3, -1]
snap_labels  = ["Start", "Early", "Late", "End"]

for ax, idx, label in zip(axes, snap_indices, snap_labels):
    ax.imshow(frames[idx])
    ax.set_title(f"{label} (step {idx % len(frames)})", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("lewm_pusht_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lewm_pusht_snapshots.png")

In [ ]:
# ── Side-by-side animation: rollout | goal ─────────────────────────────────────
EVERY_N = 4   # subsample frames for GIF speed

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("LeWM Push-T  |  left: agent rollout  |  right: goal",
             fontsize=12, fontweight="bold")

im_l = ax_l.imshow(frames[0])
im_r = ax_r.imshow(goal_frame)
for ax, t in zip([ax_l, ax_r], ["Agent rollout", "Goal"]):
    ax.set_title(t, fontsize=11)
    ax.axis("off")

sub    = frames[::EVERY_N]
stxt   = ax_l.text(0.5, -0.05, "", transform=ax_l.transAxes,
                   ha="center", fontsize=9, color="dimgray")

def _update(i):
    im_l.set_data(sub[i])
    stxt.set_text(f"step {i * EVERY_N} / {len(frames)}")
    return im_l, stxt

ani = animation.FuncAnimation(fig, _update, frames=len(sub), interval=80, blit=True)
ani.save("lewm_pusht_rollout.gif", writer="pillow", fps=12)
plt.close(fig)

display(Image("lewm_pusht_rollout.gif"))
print("Saved: lewm_pusht_rollout.gif")

## 10 · Latent space visualisation (t-SNE)

Replicates Figure 5 from the paper: the learned latent space preserves
spatial neighbourhood structure of the Push-T scene.

In [ ]:
from sklearn.manifold import TSNE

all_frames = frames[::2]
embeddings = []

with torch.no_grad():
    for frame in all_frames:
        z = encode_frame(frame)           # (1, D) on DEVICE
        # .cpu().float() — safe on MPS (no float64)
        embeddings.append(z.cpu().float().numpy())

Z    = np.concatenate(embeddings, axis=0)   # (N, D)
perp = min(30, Z.shape[0] - 1)

print(f"Running t-SNE on {Z.shape[0]} embeddings (dim {Z.shape[1]}) ...")
Z_2d = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000).fit_transform(Z)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(Z_2d[:, 0], Z_2d[:, 1],
                c=np.linspace(0, 1, len(Z_2d)), cmap="plasma", s=30, alpha=0.85)
plt.colorbar(sc, ax=ax, label="time (normalised)")
ax.set_title(
    "t-SNE of LeWM latent embeddings — Push-T rollout\n"
    "(colour = time; structure = learned scene topology)",
    fontsize=11,
)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.savefig("lewm_pusht_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lewm_pusht_tsne.png")

## 11 · Summary

| Component | Details |
|:----------|:--------|
| **Encoder** | ViT → 192-d CLS token per frame |
| **Projector** | BatchNorm-MLP → latent space |
| **ARPredictor** | Transformer + AdaLN-zero, conditioned on action embeddings |
| **SIGReg** | Sketched Isotropic Gaussian regulariser — prevents latent collapse |
| **CEM** | Cross-Entropy Method: optimise action sequences in latent space |
| **Loss terms** | Only 2: prediction MSE + λ·SIGReg |

**Key paper result:** LeWM is **48× faster** than DINO-WM at planning (~1 s vs 47 s)
because a single 192-d CLS token is ~200× more compact than DINO patch tokens.
On an M-series Mac, MPS acceleration brings inference close to what a small GPU would give.

---

### MPS changes made vs the original repo

| Issue | Fix |
|:------|:----|
| `device="cuda"` hardcoded in `SIGReg.forward()` | Monkey-patched to use `tensor.device` |
| `MUJOCO_GL="egl"` (Linux only) | Changed to `"glfw"` (macOS) |
| `torch.load(..., map_location="cuda")` | Load to CPU, then `.to(DEVICE)` |
| Implicit float64 from numpy | `transforms.ToDtype(torch.float32)` + `.float()` after load |
| CEM tensors born on CPU and moved | Created directly with `device=DEVICE` |

---
*Based on [lucas-maes/le-wm](https://github.com/lucas-maes/le-wm) · paper arXiv:2603.19312*